# 实验五：NPU 后处理集成与异构卸载

本章在实验四已完成 `YoloNmsCustom` 自定义 NMS 算子开发、编译和安装的基础上，进一步将该算子接入真实 YOLO 推理流程。实验四主要使用小规模构造数据验证自定义算子本身是否可编译、可安装、可通过 ACLNN 调用；而本章开始使用 bus.jpg 作为输入图片，通过 PyACL 加载 YOLO OM 模型完成真实推理，并从模型输出中解析候选框、置信度和类别信息。
YOLO OM 模型对一张图片会输出大量候选预测，例如形状为 (1, 25200, 85) 的结果。经过置信度阈值过滤、坐标格式转换和分数排序后，本章将真实候选框数据传入实验四开发的 YoloNmsCustom 算子，由 NPU 执行 NMS 后处理。同时，本章保留 CPU NMS 作为参考结果，对比自定义算子的 keep 和 count 输出，验证 NPU 后处理结果是否与 CPU 基准一致。

## 1. 本实验要验证什么

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">环节</th>
      <th style="text-align: left;">当前实验的要求</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">OM 推理</td>
      <td style="text-align: left;">使用 PyACL 加载 YOLO OM，得到真实模型输出 <code>pred_real</code>。如果进入 dry-run，只能说明流程可走，不能作为最终验收。</td>
    </tr>
    <tr>
      <td style="text-align: left;">NMS 输入</td>
      <td style="text-align: left;">从 YOLO 输出中拆出 <code>boxes: [N,4] float32</code> 和 <code>scores: [N] float32</code>；由于当前 Ascend C kernel 是顺序扫描版 NMS，需要先按 score 降序重排后保存到 <code>outputs/yolo_nms_inputs.npz</code>。</td>
    </tr>
    <tr>
      <td style="text-align: left;">NPU 后处理</td>
      <td style="text-align: left;">调用 <code>cd src/operators/ascendc/YoloNmsAclNNInvocation &amp;&amp; bash run.sh</code>，由 ACLNN 调用 <code>YoloNmsCustom</code>。</td>
    </tr>
    <tr>
      <td style="text-align: left;">正确性</td>
      <td style="text-align: left;">自定义算子输出的 <code>keep/count</code> 与同一份重排输入上的 CPU NMS 参考结果一致，终端出现 <code>PASSED</code>。</td>
    </tr>
  </tbody>
</table>

注意：这里比较的 `keep` 是相对于“送入自定义算子的 NMS 输入”的索引。如果后续要还原到原始 YOLO 输出行号，使用本实验保存的 `sorted_to_model_index` 做映射。


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import numpy as np

ROOT = Path('/home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend')
CFG_PATH = ROOT / 'src/configs/yolo_edge.yaml'
IMAGE_PATH = ROOT / 'src/data/images/bus.jpg'
NPZ_PATH = ROOT / 'outputs/yolo_nms_inputs.npz'
RUNNER_DIR = ROOT / 'src/operators/ascendc/YoloNmsAclNNInvocation'
CANN_ROOT = Path('/usr/local/Ascend/ascend-toolkit/8.0.RC1')

print('ROOT:', ROOT)
print('config:', CFG_PATH)
print('sample image:', IMAGE_PATH)
print('nms input npz:', NPZ_PATH)
print('ACLNN runner:', RUNNER_DIR)
print('CANN:', CANN_ROOT)


## 2. 检查 CANN 和自定义算子是否已安装

这里检查的是实验四的产物。如果找不到 `aclnn_yolo_nms_custom.h` 或 `YoloNmsCustom` 的 `.o/.json`，说明需要回到实验四重新编译并安装 `custom_opp_ubuntu_aarch64.run`。


In [ ]:
%%bash
set -e
source /usr/local/Ascend/ascend-toolkit/8.0.RC1/aarch64-linux/script/set_env.sh
export ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1

echo "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}"
which opc
which msopgen || true

find /usr/local/Ascend/ascend-toolkit/8.0.RC1/opp/vendors/customize   \( -name 'aclnn_yolo_nms_custom.h' -o -name '*YoloNmsCustom*' -o -name '*yolo_nms_custom*' \)   2>/dev/null | head -30


## 3. 用 PyACL 得到 YOLO OM 输出

这一步沿用前面实验的 `PyAclYoloSession`。如果终端提示 `using dry-run output`，表示没有真正执行 OM 推理，后面的自定义算子仍然可以跑通流程，但不能作为最终实验结论。


In [ ]:
os.chdir(ROOT)
if str(ROOT / 'src/scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src/scripts'))

from pyacl_yolo_infer import load_config, preprocess, PyAclYoloSession

cfg = load_config(CFG_PATH)
input_tensor = preprocess(str(IMAGE_PATH), cfg)

with PyAclYoloSession(cfg) as session:
    pred_real = session.infer(input_tensor)
    used_dry_run = bool(session.dry_run)

print('input tensor:', input_tensor.shape, input_tensor.dtype)
print('model output:', pred_real.shape, pred_real.dtype)
print('used dry-run:', used_dry_run)

if used_dry_run:
    print('注意：当前不是最终验收结果。请确认 OM 文件存在、PyACL 可用，并重新运行本单元。')


## 4. 从 YOLO 输出整理 NMS 输入

自定义算子当前实现的是一个清晰可验证的 NMS 核心：输入 `boxes` 和 `scores`，输出保留索引 `keep` 与数量 `count`。它的简化假设是：输入已经按照置信度从高到低排列，因此 kernel 内部只做顺序扫描和 IoU 抑制，不再做 score sort。

因此这里先在 Python 侧完成阈值过滤、类别得分合成、`xywh -> xyxy` 转换，并按 `scores` 降序重排。这样 CPU 参考结果与 NPU 自定义算子的索引语义一致。

注意：当前 `YoloNmsAclNNInvocation/src/main.cpp` 已按实验五样例固定为 `boxesShape={51,4}`、`scoresShape={51}`。如果过滤后数量不是 51，需要同步修改 runner 的 shape，或者调整输入样例。

下面直接给出调整方案，同学们可以通过日志查看过滤后的输出。

In [ ]:
from pathlib import Path
import re
import shutil
from benchmark_postprocess import xywh_to_xyxy, cpu_nms

pp = cfg['postprocess']
raw = np.asarray(pred_real, dtype=np.float32)

if raw.ndim == 3:
    raw = raw[0]
if raw.ndim != 2 or raw.shape[1] < 6:
    raise ValueError(f'expected YOLO output [N, 5 + classes], got {pred_real.shape}')

boxes_all = xywh_to_xyxy(raw[:, :4])
obj = raw[:, 4]
cls_scores = raw[:, 5:]
class_ids_all = cls_scores.argmax(axis=1).astype(np.int32)
scores_all = obj * cls_scores[np.arange(raw.shape[0]), class_ids_all]

score_threshold = float(pp['score_threshold'])
iou_threshold = float(pp['nms_iou_threshold'])
max_output = int(pp['max_detections'])

mask = scores_all >= score_threshold
model_indices = np.flatnonzero(mask).astype(np.int32)
boxes_filtered = np.asarray(boxes_all[mask], dtype=np.float32)
scores_filtered = np.asarray(scores_all[mask], dtype=np.float32)
class_ids_filtered = np.asarray(class_ids_all[mask], dtype=np.int32)

order = scores_filtered.argsort()[::-1].astype(np.int32)
boxes = np.ascontiguousarray(boxes_filtered[order], dtype=np.float32)
scores = np.ascontiguousarray(scores_filtered[order], dtype=np.float32)
class_ids = np.ascontiguousarray(class_ids_filtered[order], dtype=np.int32)
sorted_to_model_index = np.ascontiguousarray(model_indices[order], dtype=np.int32)

keep_ref = cpu_nms(boxes, scores, iou_threshold, max_output).astype(np.int32)
keep_ref_model_index = sorted_to_model_index[keep_ref]

candidate_count = int(boxes.shape[0])

print('score threshold:', score_threshold)
print('iou threshold:', iou_threshold)
print('max output:', max_output)
print('filtered boxes before sort:', boxes_filtered.shape, boxes_filtered.dtype)
print('filtered candidate count:', candidate_count)
print('boxes for custom op:', boxes.shape, boxes.dtype)
print('scores for custom op:', scores.shape, scores.dtype)
print('top scores first 10:', scores[:10])
print('cpu keep count:', len(keep_ref))
print('cpu keep first 20, sorted-input indices:', keep_ref[:20])
print('cpu keep first 20, model-row indices:', keep_ref_model_index[:20])

if candidate_count <= 0:
    raise RuntimeError('过滤后候选框数量为 0，请检查 score_threshold 或 OM 输出。')

runner_dir = ROOT / 'src/operators' / 'ascendc' / 'YoloNmsAclNNInvocation'
main_cpp = runner_dir / 'src' / 'main.cpp'

if not main_cpp.exists():
    raise FileNotFoundError(
        f'没有找到 {main_cpp}。请先完成实验四第 9 步，创建 YoloNmsAclNNInvocation 调用验证工程。'
    )

backup_cpp = main_cpp.with_suffix('.cpp.bak_before_exp5_shape')
if not backup_cpp.exists():
    shutil.copy2(main_cpp, backup_cpp)

text = main_cpp.read_text(encoding='utf-8')

replacements = [
    (
        r'std::vector<int64_t>\s+boxesShape\s*=\s*\{[^}]+\};',
        f'std::vector<int64_t> boxesShape = {{{candidate_count}, 4}};'
    ),
    (
        r'std::vector<int64_t>\s+scoresShape\s*=\s*\{[^}]+\};',
        f'std::vector<int64_t> scoresShape = {{{candidate_count}}};'
    ),
    (
        r'std::vector<int64_t>\s+keepShape\s*=\s*\{[^}]+\};',
        f'std::vector<int64_t> keepShape = {{{max_output}}};'
    ),
    (
        r'std::vector<int64_t>\s+countShape\s*=\s*\{[^}]+\};',
        'std::vector<int64_t> countShape = {1};'
    ),
]

for pattern, replacement in replacements:
    text_new, n = re.subn(pattern, replacement, text)
    if n != 1:
        raise RuntimeError(f'修改 main.cpp 失败，未唯一匹配到：{pattern}')
    text = text_new

main_cpp.write_text(text, encoding='utf-8')

print()
print('已自动同步 ACLNN runner shape:')
print('main.cpp:', main_cpp)
print(f'boxesShape = {{{candidate_count}, 4}}')
print(f'scoresShape = {{{candidate_count}}}')
print(f'keepShape = {{{max_output}}}')
print('countShape = {1}')

## 5. 保存 ACLNN runner 的输入文件

`YoloNmsAclNNInvocation/scripts/gen_data.py` 会读取这个 `.npz`，生成 `input/input_x.bin` 和 `input/input_y.bin`。`verify_result.py` 会读取同一个 `.npz` 里的 CPU 参考结果做对齐。

这里保存的 `keep_ref` 是相对于排序后 `boxes/scores` 的索引，用于和自定义算子输出直接比较；`keep_ref_model_index` 则用于追溯这些框在原始 YOLO 输出中的行号。


In [ ]:
NPZ_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    NPZ_PATH,
    boxes=boxes,
    scores=scores,
    class_ids=class_ids,
    order=order,
    sorted_to_model_index=sorted_to_model_index,
    keep_ref=keep_ref.astype(np.int32),
    keep_ref_model_index=keep_ref_model_index.astype(np.int32),
    count_ref=np.asarray([len(keep_ref)], dtype=np.int32),
    iou_threshold=np.asarray([iou_threshold], dtype=np.float32),
    max_output=np.asarray([max_output], dtype=np.int32),
)

print('saved:', NPZ_PATH)
print('file size:', NPZ_PATH.stat().st_size, 'bytes')


## 6. 检查 ACLNN 调用工程

这个工程来自官方 `AclNNInvocation` 样例，并已经改成调用 `aclnnYoloNmsCustomGetWorkspaceSize` 和 `aclnnYoloNmsCustom`。如果下面检查失败，需要回到实验四，把 `YoloNmsAclNNInvocation` 放到当前路径，并确认 `src/main.cpp`、`src/op_runner.cpp`、`scripts/gen_data.py`、`scripts/verify_result.py` 都已按当前算子接口修改。


In [ ]:
required_runner_files = [
    RUNNER_DIR / 'run.sh',
    RUNNER_DIR / 'src/main.cpp',
    RUNNER_DIR / 'src/op_runner.cpp',
    RUNNER_DIR / 'scripts/gen_data.py',
    RUNNER_DIR / 'scripts/verify_result.py',
]

for path in required_runner_files:
    print(('OK   ' if path.exists() else 'MISS '), path)

missing = [p for p in required_runner_files if not p.exists()]
if missing:
    raise FileNotFoundError('ACLNN 调用工程不完整，请先补齐上面 MISS 的文件。')


## 7. 执行 NPU 自定义 NMS

`run.sh` 会生成输入、编译调用程序、执行 ACLNN、自定义算子输出 `output_keep.bin` 和 `output_count.bin`。本实验以终端出现 `PASSED` 作为最直接的正确性信号。


In [ ]:
cmd = 'source /usr/local/Ascend/ascend-toolkit/8.0.RC1/aarch64-linux/script/set_env.sh && export ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1 && bash run.sh'
start = time.perf_counter()
proc = subprocess.run(
    ['bash', '-lc', cmd],
    cwd=str(RUNNER_DIR),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=300,
)
elapsed_ms = (time.perf_counter() - start) * 1000.0
print(proc.stdout)
print(f'runner wall time: {elapsed_ms:.3f} ms')

if proc.returncode != 0:
    raise RuntimeError(f'ACLNN runner failed, return code={proc.returncode}')
if 'PASSED' not in proc.stdout:
    print('警告：runner 返回码为 0，但没有看到 PASSED，请检查 verify_result.py 的输出。')


## 8. 读取 NPU 输出并与 CPU 参考结果对齐

这一步直接读 runner 生成的二进制输出，避免只看日志。`count` 和前 `count` 个 `keep` 索引必须与 CPU 参考结果一致。

如果需要回到原始 YOLO 输出行号，使用 `sorted_to_model_index[keep_custom]` 做映射。


In [ ]:
keep_path = RUNNER_DIR / 'output/output_keep.bin'
count_path = RUNNER_DIR / 'output/output_count.bin'

if not keep_path.exists() or not count_path.exists():
    raise FileNotFoundError('没有找到 output_keep.bin 或 output_count.bin，请先确认 run.sh 是否执行成功。')

keep_raw = np.fromfile(keep_path, dtype=np.int32)
count_arr = np.fromfile(count_path, dtype=np.int32)
if count_arr.size == 0:
    raise RuntimeError('output_count.bin 为空。')

count_custom = int(count_arr[0])
keep_custom = keep_raw[:count_custom]
keep_custom_model_index = sorted_to_model_index[keep_custom] if count_custom > 0 else np.asarray([], dtype=np.int32)

same_count = count_custom == len(keep_ref)
same_keep = np.array_equal(keep_custom, keep_ref[:count_custom])

print('custom count:', count_custom)
print('cpu count:', len(keep_ref))
print('custom first 20, sorted-input indices:', keep_custom[:20])
print('cpu first 20, sorted-input indices:', keep_ref[:20])
print('custom first 20, model-row indices:', keep_custom_model_index[:20])
print('cpu first 20, model-row indices:', keep_ref_model_index[:20])
print('same count:', same_count)
print('same keep:', same_keep)

assert same_count and same_keep, 'NPU 自定义 NMS 输出与 CPU 参考结果不一致。'
print('实验五验证通过：PyACL 输出已经接入 ACLNN 自定义 NMS，NPU 后处理结果与 CPU 参考一致。')


## 9. 本章小结

到这里，实验五的核心目标已经完成：YOLO 模型推理仍由 PyACL 加载 OM 执行，后处理中的 NMS 核心通过 `YoloNmsCustom` 卸载到 NPU，并且自定义算子输出与 CPU 参考结果一致。

PyACL 已成功加载 YOLO OM 得到真实输出；后处理 NMS 输入经 score 降序整理后，通过 ACLNN 调用 Ascend C 自定义算子 YoloNmsCustom 在 NPU 上执行；自定义算子输出的 keep/count 与 CPU NMS 参考结果一致，功能验证通过。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) 本节“卸载后处理”的含义最准确的是？
   - A. 把 YOLO NMS 从 CPU 侧迁移到 NPU 自定义算子执行
   - B. 把图片上传到 GitCode
   - C. 把 OM 转回 ONNX
   - D. 删除 CPU baseline

2. (单选题) ACLNN runner 的输入文件主要来自哪里？
   - A. PyACL 得到的 YOLO OM 输出整理后的 boxes/scores
   - B. PPT 模板
   - C. Git 提交日志
   - D. NPU 温度表

3. (单选题) 为什么真实流程中候选框数量可能变成 18866 一类的大数？
   - A. 真实 YOLO 输出过滤后候选框很多
   - B. NPU 自动复制图片
   - C. Git LFS 重复上传
   - D. CANN 把 batch 扩大为 18866

4. (单选题) 如果 `YoloNmsAclNNInvocation/src/main.cpp` 中固定 shape 为 51，但实际 boxes 是 18866，会发生什么？
   - A. shape 不匹配，编译或运行验证流程会失败
   - B. 自动变快
   - C. 自动降为 CPU
   - D. OM 文件被删除

5. (多选题) NPU 后处理集成需要对齐哪些信息？
   - A. boxes shape
   - B. scores shape
   - C. keep/count 输出
   - D. CPU baseline 的 keep/count

6. (多选题) 保存给 ACLNN runner 的输入文件时，通常需要保证什么？
   - A. float32 dtype
   - B. 内存连续
   - C. boxes 与 scores 顺序一致
   - D. 和 main.cpp 中 tensor desc shape 匹配

7. (多选题) 读取 NPU 输出后，常见的正确性检查包括哪些？
   - A. custom count 是否等于 CPU count
   - B. custom keep 是否等于 CPU keep
   - C. 映射回 model row indices 是否一致
   - D. 只看 runner 返回码是否为 0

8. (判断题) 实验五已经不再使用 OM 输出，只是在跑随机小样例。

9. (判断题) Python wrapper `import yolo_nms_custom` 是本实验后续验证的必要条件。

10. (填空题) 本节保存 ACLNN runner 输入的中间文件通常是 `outputs/____`。

11. (填空题) NPU NMS 输出中，`count` 表示 `____`。

12. (简答题) 实验四的小样例验证和实验五真实 YOLO 输出验证是什么关系？

13. (简答题) 为什么实验五需要把候选框先按 score 降序排序？

14. (简答题) runner 返回码为 0 但没有 PASSED，应该怎么判断问题？

15. (代码设计题) 写一段 Python 代码，根据 boxes 数量生成 main.cpp 中需要的 shape 数值。

> 参考答案见 answer/04.06_npu_postprocess_integration_offload_answer.ipynb。
